# Motor Insurance Analytics - Analysis Notebook

## 1. Business Understanding
Management wants to understand policy activity, premium, claims, severity, settlement speed and risk.

**Business questions:** policy status mix - premium by policy type - claim counts by status - total/average claim - claims by vehicle - region frequency and severity - customer segments - settlement time - claim types - premium vs claim cost - claim ratio by policy type - age / vehicle groups - what needs attention.

## 2. Import Libraries

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_all_data
from src.validation import validate_all_tables, validate_relationships
from src.data_cleaner import clean_data
from src import insurance_analysis as ia
from src import insights as ins

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 50)

## 3. Load Related Tables

In [ ]:
raw = load_all_data(ROOT / "data" / "raw")
for name, df in raw.items():
    print(f"{name:10s} rows={len(df):>6,}  columns={df.shape[1]}")

## 4. Understand Table Relationships
```
customers 1 --< policies >-- 1 vehicles
                policies 1 --< claims 1 --< payments
```
A claim is linked to **policy_id** because a claim is only valid when it is made against a policy that covers the vehicle.

In [ ]:
print(validate_relationships(raw))
print("Policies per customer :", raw["policies"].groupby("customer_id").size().describe()[["min","mean","max"]].round(2).to_dict())
print("Claims per policy     :", raw["claims"].groupby("policy_id").size().describe()[["min","mean","max"]].round(2).to_dict())

## 5. Basic Data Understanding

In [ ]:
for name, df in raw.items():
    print("=" * 20, name, df.shape)
    display(df.head(3)) if "display" in globals() else print(df.head(3))
    print(df.dtypes.to_dict())

## 6. Data Quality Analysis

In [ ]:
report = validate_all_tables(raw)
for table, result in report.items():
    if isinstance(result, dict):
        print(f"{table:14s} passed={result['passed']}  issues={result['issues']}")
print()
for name, df in raw.items():
    miss = df.isna().sum()
    print(name, "missing:", miss[miss > 0].to_dict(), "| duplicate rows:", int(df.duplicated().sum()))

**Reading the findings:** missing `approval_date` / `settlement_date` are *normal* (pending / rejected claims have none). Policies whose end date is before the start date are real errors - they are fixed in cleaning, not deleted.

## 7. Data Cleaning

In [ ]:
cleaned = clean_data(raw)
print(pd.DataFrame({"raw_rows": {k: len(v) for k, v in raw.items()},
                    "cleaned_rows": {k: len(v) for k, v in cleaned.items()}}))

## 8. Derived Columns

In [ ]:
print(cleaned["claims"][["claim_id", "claim_processing_days", "claim_settlement_days", "claim_to_premium_ratio",
                         "claim_to_coverage_ratio", "paid_claim_ratio"]].head())
print(cleaned["policies"][["policy_id", "policy_duration_days", "policy_active_flag"]].head(3))
print(cleaned["vehicles"][["vehicle_id", "vehicle_year", "vehicle_age"]].head(3))
pm = ia.build_policy_master(cleaned)      # one row per policy  (+ customer, vehicle, claim totals)
cm = ia.build_claims_master(cleaned)      # one row per claim   (+ policy, customer, vehicle, paid amount)
print(pm.shape, cm.shape)

## 9. EDA - Univariate

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(20, 8))
sns.histplot(pm["premium_amount"], bins=40, ax=ax[0, 0]).set_title("Premium distribution")
sns.histplot(cm["claim_amount"], bins=40, ax=ax[0, 1]).set_title("Claim amount distribution")
sns.countplot(x=pm["vehicle_age"], ax=ax[0, 2]).set_title("Vehicle age")
sns.histplot(pm["age"], bins=20, ax=ax[0, 3]).set_title("Customer age")
sns.countplot(x=pm["policy_status"], ax=ax[1, 0]).set_title("Policy status")
sns.countplot(x=cm["claim_status"], ax=ax[1, 1]).set_title("Claim status")
sns.countplot(y=cm["claim_type"], ax=ax[1, 2]).set_title("Claim type")
sns.countplot(x=pm["vehicle_type"], ax=ax[1, 3]).set_title("Vehicle type (only one type in this data)")
plt.tight_layout(); plt.show()

## 10. EDA - Bivariate

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(20, 10))
sns.boxplot(data=pm, x="policy_type", y="premium_amount", ax=ax[0, 0]).set_title("Premium vs policy type")
sns.barplot(data=cm, x="vehicle_make", y="claim_amount", ax=ax[0, 1]).set_title("Avg claim amount vs vehicle make")
sns.lineplot(data=cm, x="vehicle_age", y="claim_amount", marker="o", ax=ax[0, 2]).set_title("Vehicle age vs claim amount")
sns.barplot(data=pm, x="age_group", y="claim_count", order=ia.AGE_LABELS, ax=ax[1, 0]).set_title("Customer age group vs claims per policy")
state_avg = cm.groupby("state")["claim_amount"].mean().sort_values()
state_avg.plot.barh(ax=ax[1, 1], title="Region (state) vs avg claim amount")
sns.boxplot(data=cm.dropna(subset=["claim_processing_days"]), x="claim_status", y="claim_processing_days", ax=ax[1, 2]).set_title("Claim status vs processing days")
plt.tight_layout(); plt.show()

## 11. EDA - Multivariate

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(20, 12))
t = pm.groupby(["policy_type", "vehicle_make"]).agg(c=("claim_amount_total", "sum"), p=("premium_amount", "sum"))
sns.heatmap((t["c"] / t["p"]).unstack(), annot=True, fmt=".1f", cmap="RdYlGn_r", ax=ax[0, 0])
ax[0, 0].set_title("Policy type + vehicle make -> claim-to-premium ratio")
sns.heatmap(cm.pivot_table(index="state", columns="claim_type", values="claim_amount", aggfunc="mean"),
            cmap="YlOrRd", ax=ax[0, 1]); ax[0, 1].set_title("Region + claim type -> avg claim amount")
sns.heatmap(cm.pivot_table(index="vehicle_age_group", columns="vehicle_make", values="claim_amount", aggfunc="mean"),
            annot=True, fmt=".0f", cmap="Blues", ax=ax[1, 0]); ax[1, 0].set_title("Vehicle age + make -> avg claim amount")
sns.heatmap(pm.pivot_table(index="age_group", columns="policy_type", values="premium_amount", aggfunc="mean"),
            annot=True, fmt=".0f", cmap="Purples", ax=ax[1, 1]); ax[1, 1].set_title("Customer age + policy type -> avg premium")
plt.tight_layout(); plt.show()

## 12. Statistical Analysis

In [ ]:
print(ia.calculate_descriptive_statistics(pm, cm))
num = cm[["claim_amount", "premium_amount", "coverage_amount", "vehicle_value", "vehicle_age", "age", "claim_settlement_days"]]
plt.figure(figsize=(8, 6)); sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation of numeric variables (claims)"); plt.show()
print(cm.groupby("claim_type")["claim_amount"].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False))

## 13. KPI Analysis

In [ ]:
pay = cleaned["payments"]
kpis = {
    "Total policies": ia.calculate_total_policies(pm), "Active": ia.calculate_active_policies(pm),
    "Expired": ia.calculate_expired_policies(pm), "Cancelled": ia.calculate_cancelled_policies(pm),
    "Renewed": ia.calculate_renewed_policies(pm), "Total premium": ia.calculate_total_premium(pm),
    "Average premium": ia.calculate_average_premium(pm), "Average coverage": ia.calculate_average_coverage(pm),
    "Total claims": ia.calculate_total_claims(cm), "Approved (incl. settled)": ia.calculate_approved_claims(cm),
    "Rejected": ia.calculate_rejected_claims(cm), "Pending": ia.calculate_pending_claims(cm),
    "Total claim amount": ia.calculate_total_claim_amount(cm), "Average claim": ia.calculate_average_claim_amount(cm),
    "Approval rate %": ia.calculate_claim_approval_rate(cm), "Rejection rate %": ia.calculate_claim_rejection_rate(cm),
    "Avg processing days": ia.calculate_average_claim_processing_days(cm),
    "Avg settlement days": ia.calculate_average_claim_settlement_days(cm),
    "Total payment": ia.calculate_total_payment_amount(pay), "Average payment": ia.calculate_average_payment_amount(pay),
    "Claim-to-premium": ia.calculate_claim_to_premium_ratio(cm, pm),
    "Claim-to-coverage": ia.calculate_claim_to_coverage_ratio(cm, pm), "Paid claim ratio": ia.calculate_paid_claim_ratio(pay, cm),
}
pd.Series(kpis).to_frame("value")

## 14. Business Questions

In [ ]:
print("1. Policy status:\n", pm["policy_status"].value_counts().to_dict())
print("\n2. Premium by policy type:\n", ia.calculate_premium_by_policy_type(pm))
print("\n3. Claims by status:\n", ia.analyze_claim_status(cm))
print("\n4. Claim types:\n", ia.analyze_claim_type(cm))
print("\n5. Damage severity:\n", ia.analyze_damage_severity(cm))
print("\n6. Vehicle make:\n", ia.analyze_vehicle_claims(cm, "vehicle_make"))
print("\n7. Top 5 states by claim frequency:\n", ia.analyze_region(cm).sort_values("total_claims", ascending=False).head(5))
print("\n8. Top 5 states by severity:\n", ia.analyze_region(cm).sort_values("avg_claim_amount", ascending=False).head(5))
print("\n9. Customer age groups:\n", ia.analyze_customer_claims(cm, "age_group"))
print("\n10. Settlement time by claim type:\n", ia.analyze_claim_type(cm)[["claim_type", "avg_settlement_days"]])
print("\n11. Claim ratio by policy type:\n", ia.analyze_policy_type(pm)[["policy_type", "claim_to_premium_ratio"]])
print("\n12. Vehicle age groups:\n", ia.analyze_vehicle_risk_patterns(pm, "vehicle_age_group"))

## 15. Visualizations
The dashboard charts (Plotly) are shown here too. They need `pip install plotly`.

In [ ]:
try:
    from src import visualization as viz
    for fig in [viz.plot_damage_severity_distribution(cm), viz.plot_claim_status(cm), viz.plot_claim_to_premium_by_policy_type(pm)]:
        fig.show()
except ImportError:
    print("plotly is not installed - run: pip install plotly")

## 16. Patterns

In [ ]:
data = {"policies": pm, "claims": cm, "payments": pay}
for card in ins.generate_risk_patterns(data):
    print("PATTERN        :", card["pattern"]); print("EVIDENCE       :", card["evidence"])
    print("BUSINESS MEANING:", card["business_meaning"]); print("RECOMMENDATION :", card["recommendation"]); print("-" * 80)

## 17. Business Insights

In [ ]:
for func in (ins.generate_portfolio_insights, ins.generate_claim_insights, ins.generate_premium_insights,
             ins.generate_customer_insights, ins.generate_vehicle_insights):
    for card in func(data):
        print(f"[{card['area']}] {card['pattern']}\n    evidence: {card['evidence']}\n")

## 18. Recommendations

In [ ]:
for card in ins.generate_business_recommendations(data):
    print(card["pattern"]); print("   ->", card["recommendation"])

## 19. Final Conclusion
Write your own conclusion here in 5-6 sentences: what the portfolio looks like, which segments cost the most, what you recommend first, and one limitation (for example: *claim_amount is the amount requested, not the amount paid*).